# 06 Predict any race
Set `YEAR` and `RACE` below, then run all cells.
- `RACE` can be a name ("Mexico", "Austin", "Monza", fuzzy matched) or a round number.
- Run it after qualifying. The model trains on every race before this one and uses the final model from notebook 04 (logistic regression v2, C=0.3).
- Past races also show the actual finishing order.

2026 is a new rules era. Form features use the last 5 races, so mid-season they come only from 2026.

In [4]:
YEAR = 2025
RACE = "Australia"
GRID = None  # official grid if penalties are announced, e.g. {"VER": 1, "NOR": 2, ...}

In [5]:
import sys
from pathlib import Path

import fastf1

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
fastf1.set_log_level("ERROR")

from src.predict import predict

prediction = predict(YEAR, RACE, GRID)
print(prediction.attrs["race"])
print("Predicted podium:", ", ".join(prediction["Abbreviation"].head(3)))
if "FinishingPosition" in prediction:
    print("Actual podium:   ", ", ".join(prediction.nsmallest(3, "FinishingPosition")["Abbreviation"]))
prediction.round(3)

Australian Grand Prix 2025
Predicted podium: NOR, PIA, VER
Actual podium:    NOR, VER, RUS


,Abbreviation,Team,GridPosition,GapToPolePct,PodiumProb,FinishingPosition
0,NOR,McLaren,1.0,0.000,0.850,1.0
1,PIA,McLaren,2.0,0.112,0.663,9.0
2,VER,Red Bull,3.0,0.513,0.400,2.0
3,RUS,Mercedes,4.0,0.599,0.365,3.0
4,LEC,Ferrari,7.0,0.878,0.308,8.0
5,HAM,Ferrari,8.0,1.096,0.216,10.0
6,TSU,RB,5.0,0.764,0.044,12.0
7,GAS,Alpine,9.0,1.177,0.037,11.0
8,ALB,Williams,6.0,0.854,0.017,5.0
9,SAI,Williams,10.0,1.112,0.014,18.0


## Backtest on the season so far
Predict each finished race of `YEAR` using only data from before it, then compare to the grid baseline.

Note: the model uses **qualifying order** as the grid, while the baseline uses the **official** grid (penalties included). That gives the baseline a small edge.

In [6]:
import pandas as pd

from src.predict import load_history, predict_race, quali_rows

history = load_history()
season = history[history["Year"] == YEAR]
rows = []
for rnd in sorted(season["Round"].unique()):
    pred = predict_race(history, quali_rows(YEAR, int(rnd)))
    race = season[season["Round"] == rnd]
    podium = set(race.loc[race["Podium"] == 1, "Abbreviation"])
    rows.append({"Round": rnd, "GrandPrix": race["GrandPrix"].iloc[0],
                 "Model": len(set(pred["Abbreviation"].head(3)) & podium),
                 "Grid": len(set(race.nsmallest(3, "GridPosition")["Abbreviation"]) & podium)})

backtest = pd.DataFrame(rows)
print("Hits per race:", backtest[["Model", "Grid"]].mean().round(2).to_dict())
backtest

Hits per race: {'Model': 2.21, 'Grid': 2.25}


,Round,GrandPrix,Model,Grid
0,1,Australian Grand Prix,2,2
1,2,Chinese Grand Prix,3,3
2,3,Japanese Grand Prix,3,3
3,4,Bahrain Grand Prix,2,2
4,5,Saudi Arabian Grand Prix,2,2
5,6,Miami Grand Prix,2,1
6,7,Emilia Romagna Grand Prix,3,2
7,8,Monaco Grand Prix,3,3
8,9,Spanish Grand Prix,2,2
9,10,Canadian Grand Prix,2,2
